# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Emeka-techDev/ml-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Selected Lane: Lane 2
Reason : it helps assist in decision making on what pages needs review first and provide a reasons why the pages should be reviewed.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision the work improves?**

This work assist in prioritizing pages that need review and saving large amount of time and limited resources that would go into impractical review of every page

**Who acts on it?**

Content writer and website reviewers

**Cost of wrong recommendation?**

time and resources are lost reviewing and updating the wrong site, site that actually need update and review might be ommitted, overall visible of the platform does not change


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [12]:
import pandas as pd

url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df.head()

print(f"Unique Number of pages: {df['content_id'].nunique():,}")

df.shape
print(
    f"Declining pages: {df['is_declining'].sum():,} "
    f"out of {len(df):,}"
)

# declining_rate = df["is_declining"].mean() * 100

# print(f"Declining pages: {declining_rate:.1f}%")
print(f"Total search volume: {df['search_volume'].sum():,.0f}")

Unique Number of pages: 30,000
Declining pages: 5,563 out of 30,000
Total search volume: 4,374,350


The starter file contains 30,000 pages, which makes reviewing each page individually impractical and creates a clear need for prioritization. Of those pages, 5,563 (18%) are declining, this suggest a meaningful subset of the content require review. The dataset contain a total search volume of 4,374,350 which indicates that the content represents a substantial amount of search demand, making the question which page should be reviewed first substantial valuable.
  

**BASELINE SETUP**



In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score


# 2. CREATE TRAIN / TEST SPLIT

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features].copy()
y = df["is_declining"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")


# ============================================================
# 3. CREATE BASELINE TRAIN DATA
# ============================================================

# We use the X_train indices to select the corresponding
# complete rows from df.
#
# We cannot use X_train directly because it only contains
# the ML features. The baseline also needs columns such as
# position_tier, impressions_90d, content_id, etc.

baseline_train = df.loc[X_train.index].copy()

# 4. BASELINE SIGNAL 1
#    RECENT PERFORMANCE DECLINE

baseline_train["impression_change_pct"] = (
    (
        baseline_train["impressions_last_30d"]
        - baseline_train["impressions_prev_30d"]
    )
    / baseline_train["impressions_prev_30d"].replace(0, np.nan)
) * 100

baseline_train["decline_signal"] = (
    baseline_train["impression_change_pct"] < -20
)


# 5. BASELINE SIGNAL 2
#    DECLINING + MEANINGFUL DEMAND

baseline_train["has_demand"] = (
    baseline_train["search_volume"].fillna(0) >= 20
)

baseline_train["declining_demand_score"] = (
    baseline_train["decline_signal"]
    & baseline_train["has_demand"]
).astype(int) * 3


# 6. BASELINE SIGNAL 3
#    STALE + VISIBLE

baseline_train["stale_visible_score"] = (
    (baseline_train["days_since_last_update"] >= 91)
    & (baseline_train["impressions_90d"] > 0)
).astype(int) * 2


# 7. BASELINE SIGNAL 4
#    PAGE-ONE DECAY RISK

page_one_train = baseline_train["position_tier"].isin(
    ["top_3", "page_1", "striking"]
)

baseline_train["page_one_decay_score"] = (
    page_one_train
    & baseline_train["decline_signal"]
).astype(int) * 2


# 8. BASELINE SIGNAL 5
#    CTR REVIEW CONTEXT
#
# IMPORTANT:
# The CTR threshold is learned from TRAINING data only.

ctr_threshold = baseline_train["ctr"].quantile(0.25)

print(f"\nCTR 25th percentile from training data: {ctr_threshold:.4f}")

baseline_train["ctr_review_score"] = (
    page_one_train
    & (baseline_train["ctr"] <= ctr_threshold)
).astype(int)


# 9. CREATE BASELINE SCORE ON TRAINING DATA

baseline_train["baseline_score"] = (
    baseline_train["declining_demand_score"]
    + baseline_train["stale_visible_score"]
    + baseline_train["page_one_decay_score"]
    + baseline_train["ctr_review_score"]
)


# 10. CREATE BASELINE TEST DATA



baseline_test = df.loc[X_test.index].copy()


# 11. APPLY SIGNAL 1 TO TEST DATA

baseline_test["impression_change_pct"] = (
    (
        baseline_test["impressions_last_30d"]
        - baseline_test["impressions_prev_30d"]
    )
    / baseline_test["impressions_prev_30d"].replace(0, np.nan)
) * 100

baseline_test["decline_signal"] = (
    baseline_test["impression_change_pct"] < -20
)


# ============================================================
# 12. APPLY SIGNAL 2 TO TEST DATA
# ============================================================

baseline_test["has_demand"] = (
    baseline_test["search_volume"].fillna(0) >= 20
)

baseline_test["declining_demand_score"] = (
    baseline_test["decline_signal"]
    & baseline_test["has_demand"]
).astype(int) * 3


# ============================================================
# 13. APPLY SIGNAL 3 TO TEST DATA
# ============================================================

baseline_test["stale_visible_score"] = (
    (baseline_test["days_since_last_update"] >= 91)
    & (baseline_test["impressions_90d"] > 0)
).astype(int) * 2


# ============================================================
# 14. APPLY SIGNAL 4 TO TEST DATA
# ============================================================

page_one_test = baseline_test["position_tier"].isin(
    ["top_3", "page_1", "striking"]
)

baseline_test["page_one_decay_score"] = (
    page_one_test
    & baseline_test["decline_signal"]
).astype(int) * 2


# ============================================================
# 15. APPLY SIGNAL 5 TO TEST DATA
#
# IMPORTANT:
# We use the CTR threshold learned from TRAINING data.
# We do NOT calculate a new threshold from the test data.
# ============================================================

baseline_test["ctr_review_score"] = (
    page_one_test
    & (baseline_test["ctr"] <= ctr_threshold)
).astype(int)


# ============================================================
# 16. FINAL BASELINE SCORE ON TEST DATA
# ============================================================

baseline_test["baseline_score"] = (
    baseline_test["declining_demand_score"]
    + baseline_test["stale_visible_score"]
    + baseline_test["page_one_decay_score"]
    + baseline_test["ctr_review_score"]
)


# ============================================================
# 17. RANK TEST PAGES
# ============================================================

baseline_test_queue = baseline_test.sort_values(
    "baseline_score",
    ascending=False
)


# ============================================================
# 18. PRECISION@20
# ============================================================

baseline_top_20 = baseline_test_queue.head(20)

baseline_precision_at_20 = (
    baseline_top_20["is_declining"].mean()
)

print(
    f"\nBaseline Precision@20: "
    f"{baseline_precision_at_20:.3f}"
)


# ============================================================
# 19. AVERAGE PRECISION
# ============================================================

baseline_ap = average_precision_score(
    baseline_test_queue["is_declining"],
    baseline_test_queue["baseline_score"]
)

print(
    f"Baseline Average Precision: "
    f"{baseline_ap:.3f}"
)


# ============================================================
# 20. SHOW TOP 20 BASELINE QUEUE
# ============================================================

baseline_test_queue[
    [
        "content_id",
        "baseline_score",
        "is_declining",
        "search_volume",
        "impression_change_pct",
        "days_since_last_update",
        "position_tier",
        "ctr"
    ]
].head(20)


Training rows: 24,000
Test rows: 6,000

CTR 25th percentile from training data: 0.0000

Baseline Precision@20: 0.050
Baseline Average Precision: 0.300


,content_id,baseline_score,is_declining,search_volume,impression_change_pct,days_since_last_update,position_tier,ctr
14641,content_2fa6a97a3c70,8,False,40.0,-38.541667,104,page_1,0.0
25572,content_77a83ecf7c27,8,True,30.0,-62.820513,104,striking,0.0
29663,content_1980254ef582,8,False,70.0,-76.923077,92,striking,0.0
22730,content_bb1edbf0ba6c,8,False,20.0,-22.297297,104,striking,0.0
22336,content_3a470a752d01,8,False,20.0,-100.000000,104,striking,0.0
24012,content_d0351144852b,8,False,40.0,-79.411765,104,top_3,0.0
5732,content_b893db8d7619,8,False,50.0,-89.300412,104,page_1,0.0
28617,content_467784932d6c,8,False,30.0,-100.000000,104,striking,0.0
28269,content_8e329558bb9e,8,False,40.0,-65.748031,104,page_1,0.0
12053,content_f421fb2e10ea,8,False,5400.0,-91.044776,104,page_1,0.0


In [24]:
word_threshold = df["word_count"].quantile(0.25)

print(word_threshold)

2413.0


In [25]:
thin_visible = df[
    (df["word_count"] <= word_threshold) &
    (df["impressions_90d"] > 0)
]

not_thin_visible = df[
    (df["word_count"] > word_threshold) &
    (df["impressions_90d"] > 0)
]

print(
    "Thin + visible declining rate:",
    thin_visible["is_declining"].mean()
)

print(
    "Not thin + visible declining rate:",
    not_thin_visible["is_declining"].mean()
)

Thin + visible declining rate: 0.09881635581061693
Not thin + visible declining rate: 0.23312406576980568


### **Select Feature to use in training model**

In [26]:
print(pd.crosstab(
    df["trend_direction"],
    df["is_declining"],
    normalize="index"
))

is_declining        False     True 
trend_direction                    
down             0.657914  0.342086
flat             1.000000  0.000000
new              1.000000  0.000000
stable           1.000000  0.000000
up               1.000000  0.000000


The trend direction would be from the feature training data because the exploratory analysis showed that is is strongly aligned with the is_declining target and could leak information about the target into the model

In [27]:
print(
    df.groupby("position_tier")["is_declining"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

               count      mean
position_tier                 
page_3_5        7242  0.224938
page_1         11814  0.196716
striking        7304  0.189348
top_3           2321  0.080569
deep            1319  0.030326


There is clearly an association between search position and declining status. however avg_position contains more information and we would use it instead of position_tier


In [28]:
print(
    df.groupby("freshness_tier")["is_declining"]
    .agg(["count", "mean"])
)

                count      mean
freshness_tier                 
0-30            20480  0.159619
181+              174  0.063218
31-90             175  0.102857
91-180           9171  0.246974


Freshness appear to contain useful information, but it's relationship with decline is not simply linear (this non-linearity would make the tree-based models outperform a simple linear model)

In [29]:
print(
    df.groupby("is_declining")[
        [
            "search_volume",
            "days_since_last_update",
            "ctr",
            "avg_position",
            "word_count",
            "engagement_rate"
        ]
    ].median()
)

              search_volume  days_since_last_update   ctr  avg_position  \
is_declining                                                              
False                  10.0                    20.0  0.00          10.7   
True                   10.0                    22.0  0.16          11.2   

              word_count  engagement_rate  
is_declining                               
False             2835.0              0.0  
True              3059.0              0.0  


the search volume, ctr, avg_position and engagement_rate would be useful for business analysis and it will be useful for inclusion


### **Train model**

**STEP 1 : Create the feature (We already have the right features in the blocks above**

**STEP 2: handle missing values (using a preprocessing pipeline)**


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scalar", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

**STEP 4: Train Model**

In [32]:
model.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scalar', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [33]:
#Logisctic Regresion (lr) test score

lr_test_scores = model.predict_proba(X_test)[:, 1]

print(lr_test_scores)

[7.52793154e-01 5.78905859e-01 3.89265041e-01 ... 6.49393369e-06
 7.69069500e-01 3.47465449e-01]


In [34]:
import numpy as np

results = X_test.copy()

results["actual_declining"] = y_test.values
results["logistic_regression_model_score"] = lr_test_scores

results = results.sort_values("logistic_regression_model_score", ascending=False)
top_20 = results.head(20)

precision_at_20 = top_20["actual_declining"].mean()
print(f"Precision@20: {precision_at_20:.3f}")

baseline_top_20 = baseline_queue.head(20)
baseline_precision_at_20 = baseline_top_20["is_declining"].mean()
print(f"Baseline Precision@20: {baseline_precision_at_20:.3f}")

Precision@20: 0.300
Baseline Precision@20: 0.050


6 of the top 20 pages selected by the model were actually labelled as declining

In [35]:


lr_ap = average_precision_score(
    y_test,
    lr_test_scores
)

print(f"Logistic regression Average Precision: {lr_ap:.3f}")

#baseline average precision score
baseline_ap = average_precision_score(
    baseline_queue["is_declining"],
    baseline_queue["baseline_score"]
)

print(f"baseline Average Precision: {baseline_ap:.3f}")
#

Logistic regression Average Precision: 0.283
baseline Average Precision: 0.291


**Decision Tree model**

In [36]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', DecisionTreeClassifier(
        max_depth=5,
        random_state=42,
        class_weight="balanced"
    ))
])



In [37]:
tree_model.fit(X_train, y_train)

tree_scores = tree_model.predict_proba(X_test)[:, 1]

results["tree_model_score"] = tree_scores

results = results.sort_values("tree_model_score", ascending=False)
top_20 = results.head(20)

tree_precision_at_20 = top_20["actual_declining"].mean()
print(f"Precision@20: {tree_precision_at_20:.3f}")
print(f"Baseline Precision@20: {baseline_precision_at_20:.3f}")

Precision@20: 0.150
Baseline Precision@20: 0.050


In [38]:
tree_ap = average_precision_score(
    y_test,
    tree_scores
)

print(f"Tree regression Average Precision: {tree_ap:.3f}")
print(f"baseline Average Precision: {baseline_ap:.3f}")

Tree regression Average Precision: 0.450
baseline Average Precision: 0.291


**Forest Model**


In [39]:
from sklearn.ensemble import RandomForestClassifier

forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

forest_model.fit(X_train, y_train)

forest_scores = forest_model.predict_proba(X_test)[:, 1]

results["forest_model_score"] = forest_scores

results = results.sort_values("forest_model_score", ascending=False)
top_20 = results.head(20)

forest_precision_at_20 = top_20["actual_declining"].mean()
print(f"Precision@20: {forest_precision_at_20:.3f}")
print(f"Baseline Precision@20: {baseline_precision_at_20:.3f}")

Precision@20: 0.100
Baseline Precision@20: 0.050


In [26]:
forest_ap = average_precision_score(
    y_test,
    forest_scores
)

print(f"Forest regression Average Precision: {forest_ap:.3f}")
print(f"baseline Average Precision: {baseline_ap:.3f}")

Tree regression Average Precision: 0.537
baseline Average Precision: 0.291


The decision tree has the best score therefore we would be using it for the main ranking

In [40]:
# Start from the original test rows
model_queue = df.loc[X_test.index].copy()

model_queue["actual_declining"] = y_test

model_queue["lr_score"] = lr_test_scores
model_queue["tree_score"] = tree_scores
model_queue["forest_score"] = forest_scores

# Use the strongest current model for the main ranking
model_queue["model_score"] = model_queue["tree_score"]

model_queue = model_queue.sort_values(
    "model_score",
    ascending=False
)



In [41]:
def get_reason_codes(row):
    reasons = []

    if row["decline_signal"] and row["has_demand"]:
        reasons.append("declining_with_demand")

    if (
        row["days_since_last_update"] >= 91
        and row["impressions_90d"] > 0
    ):
        reasons.append("stale_visible")

    if (
        row["position_tier"] in ["top_3", "page_1", "striking"]
        and row["decline_signal"]
    ):
        reasons.append("page_one_decay")

    if (
        row["position_tier"] in ["top_3", "page_1", "striking"]
        and row["ctr"] <= ctr_threshold
    ):
        reasons.append("low_ctr")

    return reasons


model_queue["reason_codes"] = model_queue.apply(
    get_reason_codes,
    axis=1
)

In [42]:
def suggest_action(row):

    if (
        row["model_score"] >= 0.70
        and row["days_since_last_update"] >= 91
        and row["impressions_90d"] > 0
    ):
        return "refresh"

    if (
        row["model_score"] >= 0.70
        and row["position_tier"] in ["top_3", "page_1", "striking"]
    ):
        return "protect"

    if (
        row["model_score"] >= 0.50
        and row["search_volume"] >= 20
    ):
        return "expansion_review"

    if row["model_score"] >= 0.30:
        return "monitor"

    return "lower_priority"

In [43]:
def confidence_label(score):
    if score >= 0.70:
        return "high"
    elif score >= 0.50:
        return "medium"
    else:
        return "low"


model_queue["confidence"] = model_queue["model_score"].apply(
    confidence_label
)

In [44]:
model_queue["suggested_action"] = model_queue.apply(
    suggest_action,
    axis=1
)

model_queue["priority_confidence"] = model_queue[
    "model_score"
].apply(confidence_label)

In [45]:
action_queue = model_queue[
    [
        "content_id",
        "model_score",
        "priority_confidence",
        "suggested_action",
        "reason_codes",
        "search_volume",
        "position_tier",
        "days_since_last_update",
        "ctr",
        "avg_position"
    ]
].sort_values(
    "model_score",
    ascending=False
)

In [46]:
action_queue.head(20)

,content_id,model_score,priority_confidence,suggested_action,reason_codes,search_volume,position_tier,days_since_last_update,ctr,avg_position
18509,content_095661034f9b,0.925637,high,refresh,[stale_visible],0.0,page_3_5,104,0.00,39.4
10405,content_192488282b56,0.925637,high,refresh,[stale_visible],0.0,page_3_5,104,0.00,28.5
25872,content_4f319d8960b2,0.925637,high,refresh,[stale_visible],0.0,page_3_5,104,0.00,28.4
5779,content_617bb194934b,0.925637,high,refresh,[stale_visible],0.0,page_3_5,104,0.00,27.3
14467,content_a59988d4dd03,0.925637,high,refresh,[stale_visible],0.0,page_3_5,104,0.00,30.2
8057,content_068134334415,0.912320,high,protect,[page_one_decay],0.0,top_3,20,1.17,2.9
9158,content_a001aa4be7b7,0.912320,high,refresh,"[stale_visible, page_one_decay]",0.0,striking,106,2.62,16.2
14718,content_b0d9646900d0,0.912320,high,refresh,[stale_visible],20.0,page_1,104,0.82,4.6
29943,content_9d7d0e8e5278,0.912320,high,monitor,[],0.0,page_3_5,20,0.78,21.3
13095,content_e4768c716a36,0.912320,high,protect,[page_one_decay],10.0,page_1,20,1.01,5.3


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*


What I can claim

This analysis provides decision support for prioritizing pages for human review. The model and baseline identify pages that show characteristics associated with declining performance, such as recent performance changes, search visibility, freshness, ranking position, and CTR context.

The output is a ranked review queue, rather than an automatic recommendation to make a content change. For example, a page with a high model score and a stale_visible reason code can be prioritized for refresh review, while a page with page_one_decay can be prioritized for protection review.

The results are directional and observational. They show patterns in the starter dataset and help identify pages that may deserve attention. The reason codes make the ranking more interpretable by showing which signals contributed to a page being prioritized.

The current results also show that not every suggested signal is useful. For example, thin visible pages had a lower observed declining rate than non-thin visible pages in this dataset, so I did not use thin content as a positive declining signal in the baseline.


What I cannot claim

I cannot claim that a high-scoring page will benefit from a refresh, expansion, protection, pruning, or monitoring. The model predicts declining status/risk; it does not estimate the causal effect of taking a particular action.

I also cannot claim that the model predicts Google's rankings or Google's algorithm. Search ranking is influenced by many factors that are not represented in this dataset, and the model is only learning patterns present in the available data.

I cannot claim that a decline is guaranteed to continue, or that declining performance was caused by the factors identified by the model. The relationships observed here are associations, not causal explanations.

Finally, the ranked queue should not be treated as an automated publishing or pruning system. The top-ranked pages are candidates for human review, where additional context can be considered before deciding what action, if any, should be taken

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.